# Whisper small with the final dataset

In [1]:
# ══════════════════════════════════════════════════════
# CELLULE 1 — Installation
# ══════════════════════════════════════════════════════
!pip install -q transformers datasets accelerate \
             huggingface_hub evaluate jiwer

print("Installation terminée! ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.0 MB/s eta 0:00:00:00:01
Installation terminée! ✅


In [ ]:
# ══════════════════════════════════════════════════════
# CELLULE 2 — Initialisation
# ══════════════════════════════════════════════════════
import torch
from datasets import load_dataset
from transformers import WhisperProcessor
from huggingface_hub import login

# ── Login ─────────────────────────────────────────────
login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Nombre de GPUs: {torch.cuda.device_count()}")

# ── Charger dataset ───────────────────────────────────
print("\nChargement du dataset...")
dataset = load_dataset("abdeldjalil-smaili/algerian-medical-asr-v3")

dataset_split = dataset['train'].train_test_split(
    test_size=0.2, seed=42
)
train_data = dataset_split['train']
test_data  = dataset_split['test']

print(f"Train: {len(train_data)} samples ✅")
print(f"Test:  {len(test_data)} samples ✅")

# ── Processor ─────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="Arabic",
    task="transcribe"
)

# ── Préparer données ──────────────────────────────────
def prepare_dataset(sample):
    audio = sample['audio']
    sample['input_features'] = processor.feature_extractor(
        audio['array'],
        sampling_rate=16000
    ).input_features[0]
    sample['labels'] = processor.tokenizer(
        sample['sentence']
    ).input_ids
    return sample

print("\nPréparation des données...")
train_data = train_data.map(
    prepare_dataset,
    remove_columns=train_data.column_names
)
test_data = test_data.map(
    prepare_dataset,
    remove_columns=test_data.column_names
)
print(f"Données prêtes! ✅")

GPU: Tesla T4
Nombre de GPUs: 2

Chargement du dataset...


README.md:   0%|          | 0.00/396 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/195M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2237 [00:00<?, ? examples/s]

Train: 1789 samples ✅
Test:  448 samples ✅


preprocessor_config.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)


Préparation des données...


Map:   0%|          | 0/1789 [00:00<?, ? examples/s]

Map:   0%|          | 0/448 [00:00<?, ? examples/s]

Données prêtes! ✅


In [5]:
import torch
torch.cuda.empty_cache()
del model

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(device)
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-algerian-final",
    per_device_train_batch_size=4,   # 4 × 2 GPUs = 8 effectif
    gradient_accumulation_steps=2,   # 8 × 2 = 16 effectif total
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("="*50)
print(f"Fine-tuning: {len(train_data)} samples")
print(f"Batch effectif: 4 × 2 GPUs × 2 accumulation = 16")
print(f"Max steps: 1000")
print("="*50)

trainer.train()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Fine-tuning: 1789 samples
Batch effectif: 4 × 2 GPUs × 2 accumulation = 16
Max steps: 1000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
200,0.591865,0.172269,17.132588
400,0.130821,0.145027,13.099042
600,0.044726,0.142996,11.821086
800,0.008081,0.159556,11.182109
1000,0.003285,0.159439,10.862620


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


Sample predictions:
  Prédit    : ما مدة العلاج بالطب الصيني؟
  Référence : ما مدة العلاج بالطب الصيني
  Prédit    : كيف الوقاية من أمراض القلب
  Référence : كيف الوقاية من أمراض القلب
  Prédit    : وش هو اسبابو 
  Référence : وش هو اسبابو


... (11 outputs left out)


Sample predictions:
  Prédit    : ما مدة العلاج بالطب الصيني؟
  Référence : ما مدة العلاج بالطب الصيني
  Prédit    : كيف الوقاية من أمراض القلب
  Référence : كيف الوقاية من أمراض القلب
  Prédit    : وش هو اسبابو
  Référence : وش هو اسبابو


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1000, training_loss=0.6262162633091212, metrics={'train_runtime': 5988.6084, 'train_samples_per_second': 2.672, 'train_steps_per_second': 0.167, 'total_flos': 4.61044035551232e+18, 'train_loss': 0.6262162633091212, 'epoch': 8.928571428571429})

In [ ]:
from huggingface_hub import HfApi

# Utilise le token Write (pas le token Read)
from getpass import getpass
HF_WRITE_TOKEN = getpass("Hugging Face write token: ")

api = HfApi()

# Créer le repo
api.create_repo(
    repo_id="abdeldjalil-smaili/whisper-algerian-medical-final",
    repo_type="model",
    token=HF_WRITE_TOKEN,
    exist_ok=True
)

# Upload le modèle
api.upload_folder(
    folder_path="./whisper-algerian-final",
    repo_id="abdeldjalil-smaili/whisper-algerian-medical-final",
    repo_type="model",
    token=HF_WRITE_TOKEN
)

print("Modèle sauvegardé! ✅")
print("WER final: 10.9%")
print("🎉 Phase 1 ASR terminée!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Modèle sauvegardé! ✅
WER final: 10.9%
🎉 Phase 1 ASR terminée!
